## Extract spectra for use in ad hoc cloudy run

This code extracts a stellar and an AGN incident spectrum for the reference
cloudy models in `stellar_*/` and `agn_*/` (`1.sed`). It uses synthesizer's
test grids, `test_grid.hdf5` (SPS) and `test_grid_agn-nlr.hdf5` (AGN), read
from synthesizer's grid directory (`GRID_DIR`, i.e. `$SYNTHESIZER_DIR/grids`).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from synthesizer import GRID_DIR
from synthesizer.grid import Grid
from unyt import Msun, c, h

In [ ]:
# The path to the grids: $SYNTHESIZER_DIR/grids
grid_dir = GRID_DIR
print(grid_dir)

#### Stellar

In [ ]:
# Define the SPS grid to use
sps_model = "test_grid"

# Initialise the grid
grid = Grid(grid_name=sps_model, grid_dir=grid_dir, ignore_lines=True)

# Define the values for the grid point we want
log10age = 6.0
metallicity = 0.001

# Find the nearest grid point to these values
grid_point = grid.get_grid_point(
    log10ages=log10age, metallicities=metallicity
)

In [ ]:
# extract the sed into a synthesizer Sed object
sed = grid.get_sed_at_grid_point(grid_point, spectra_type="incident")

# plot the sed
plt.loglog(sed.lam.to("Angstrom").value, sed.lnu)
plt.show()

In [ ]:
# define frequency
nu = c / sed.lam

# define energy
E = h * nu

# extract the luminosity
lnu = sed.lnu

# set zero values to very small value
lnu[lnu == 0.0] = 1e-100

# save tabulated spectrum for use in cloudy
np.savetxt(
    "stellar_standard/1.sed",
    np.array([E.to("Ry").value[::-1], lnu.to("erg/s/Hz").value[::-1]]).T,
)
np.savetxt(
    "stellar_vacuum/1.sed",
    np.array([E.to("Ry").value[::-1], lnu.to("erg/s/Hz").value[::-1]]).T,
)

In [ ]:
# various plot of the spectrum

# plt.loglog(E.to("Ry").value[::-1], sed.lnu.to('erg/s/Hz').value[::-1])
# plt.show()

plt.loglog(E.to("eV").value[::-1], sed.lnu.to("erg/s/Hz").value[::-1])
plt.show()

# plt.loglog(sed.lam.to("Angstrom").value, sed.lnu.to('erg/s/Hz').value)
# plt.show()

#### AGN

In [ ]:
# Define the disc model grid to use
disc_model = "test_grid_agn-nlr"

# Initialise the grid
grid = Grid(grid_name=disc_model, grid_dir=grid_dir, ignore_lines=True)

In [ ]:
# Define the values of the grid point we're interested in
mass = 1e8 * Msun
accretion_rate_eddington = 0.1
cosine_inclination = 0.5

# Get grid point
grid_point = grid.get_grid_point(
    masses=mass,
    accretion_rates_eddington=accretion_rate_eddington,
    cosine_inclinations=cosine_inclination,
)

# The incident spectrum does not depend on the remaining (cloudy) axes of the
# grid, so take their first point
grid_point = tuple(0 if isinstance(i, slice) else i for i in grid_point)

print(grid_point)

In [ ]:
sed = grid.get_sed_at_grid_point(grid_point, spectra_type="incident")

plt.loglog(sed.lam.to("Angstrom").value, sed.lnu)
plt.show()

In [ ]:
# define frequency
nu = c / sed.lam

# define energy
E = h * nu


lnu = sed.lnu

lnu[lnu == 0.0] = 1e-100

# save tabulated spectrum
np.savetxt(
    "agn_standard/1.sed",
    np.array([E.to("Ry").value[::-1], lnu.to("erg/s/Hz").value[::-1]]).T,
)
np.savetxt(
    "agn_vacuum/1.sed",
    np.array([E.to("Ry").value[::-1], lnu.to("erg/s/Hz").value[::-1]]).T,
)

plt.loglog(E.to("eV").value[::-1], sed.lnu.to("erg/s/Hz").value[::-1])
plt.show()